In [1]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth

os.environ["GROQ_API_KEY"] = userdata.get("GROK_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("Bloom & Bliss Orders")

ws = sh.sheet1

ws.append_row(["name", "bouquet", "qty", "date", "occasion", "total"])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1BNOAqQO0q6S-csyPSzFKIJ0t5K1HHrr2f6TZVn3K4Y4


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {
    "red roses": 399,
    "pink roses": 499,
    "sunflower": 449,
    "mixed flowers": 649,
    "lily": 799,
    "orchid": 999
}

HOURS = {
    "monday": "9 am to 9 pm",
    "tuesday": "9 am to 9 pm",
    "wednesday": "9 am to 9 pm",
    "thursday": "9 am to 9 pm",
    "friday": "9 am to 9 pm",
    "saturday": "9 am to 9 pm",
    "sunday": "9 am to 9 pm"
}

class Order(BaseModel):
    """A flower order taken from a customer's chat message."""

    name: str = Field(description="Customer's first name")

    bouquet: Literal[
        "red roses", "pink roses", "sunflower",
        "mixed flowers", "lily", "orchid"
    ] = Field(description="Selected flower bouquet")

    qty: int = Field(description="How many bouquets", ge=1)

    date: str = Field(description="Pickup date as the customer said it")

    occasion: str = Field(description="Occasion for ordering flowers")

In [3]:
@tool
def get_price(bouquet: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of flower bouquets.

    Args:
        bouquet: Flower bouquet, e.g. "red roses", "pink roses" or "sunflower"
        qty: How many bouquets the customer wants
    """
    price = MENU.get(bouquet.lower().strip())

    if price is None:
        return f"'{bouquet}' is not on the menu. We have: {', '.join(MENU)}."

    return f"{qty} x {bouquet} = Rs {price * qty}"


@tool
def is_open(day: str) -> str:
    """Check the flower shop's opening hours for a day of the week, e.g. "sunday"."""

    hours = HOURS.get(day.lower().strip())

    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."


@tool
def log_order(name: str, bouquet: str, qty: int, date: str, occasion: str) -> str:
    """Save a confirmed flower order to the Orders sheet.
    Use only when the customer has given their name, bouquet,
    quantity, pickup date and occasion."""

    try:
        order = Order(
            name=name,
            bouquet=bouquet.lower().strip(),
            qty=qty,
            date=date,
            occasion=occasion
        )

    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."

    total = MENU[order.bouquet] * order.qty

    ws.append_row([
        order.name,
        order.bouquet,
        order.qty,
        order.date,
        order.occasion,
        total
    ])

    return f"Saved: {order.qty} x {order.bouquet} for {order.name} on {order.date}, Rs {total}."

In [4]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for Bloom & Bliss Flower Shop.
FACTS:  Orders need 24 hours' notice. Bouquets are available for birthdays,
        anniversaries, weddings and other special occasions.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, bouquet, quantity,
        pickup date and occasion; ask if missing.
        Save only confirmed orders.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

history = []

def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({
            "messages": history + [{"role": "user", "content": text}]
        })
        history = result["messages"]
        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."


print(chat("Are you open on Sunday? How much are 2 red roses bouquets?"))

print(chat("I'm cherry. Please book 2 red roses bouquets for 24 Dec for a birthday."))

print(chat("Can you tell me my order details?"))

Yes, we’re open from 9 am to 9 pm on Sundays. Two red‑rose bouquets cost Rs 798 in total.
✅ Order confirmed!  
2 × red roses for **Cherry** on **24 Dec** (birthday). Total: **Rs 798**.  
We’ll see you on the pickup day. Let us know if you need anything else!
Here’s your order:  
- **Name:** Cherry  
- **Bouquet:** 2 × red roses  
- **Pickup date:** 24 Dec  
- **Occasion:** Birthday  
- **Total:** Rs 798  

We’re all set—see you on the pickup day!
